In [276]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import pyarrow
import pathlib
import gc
import warnings
warnings.filterwarnings('ignore')

ROOT = pathlib.Path().resolve()
DATA_FOLDER = ROOT / "data/bike"

print(DATA_FOLDER)


C:\Users\PatCa\Documents\PythonScripts\test_projects\big_data\bike_data\data\bike


In [277]:
col_names = ['tripduration', 'starttime', 'stoptime', 'start station id',
             'start station name', 'start station latitude','start station longitude',
             'end station id', 'end station name', 'end station longitude',
             'end station latitude', 'bikeid','usertype','birth year','gender']

df = pd.read_csv('data/bike/csv/2019/201901-citibike-tripdata.csv', names=col_names, dtype=str)
display(df.head(5))

,tripduration,starttime,stoptime,start station id,start station name,start station latitude,start station longitude,end station id,end station name,end station longitude,end station latitude,bikeid,usertype,birth year,gender
0,tripduration,starttime,stoptime,start station id,start station name,start station latitude,start station longitude,end station id,end station name,end station latitude,end station longitude,bikeid,usertype,birth year,gender
1,320,2019-01-01 00:01:47.4010,2019-01-01 00:07:07.5810,3160,Central Park West & W 76 St,40.77896784,-73.97374737,3283,W 89 St & Columbus Ave,40.7882213,-73.97041561,15839,Subscriber,1971,1
2,316,2019-01-01 00:04:43.7360,2019-01-01 00:10:00.6080,519,Pershing Square North,40.751873,-73.977706,518,E 39 St & 2 Ave,40.74780373,-73.9734419,32723,Subscriber,1964,1
3,591,2019-01-01 00:06:03.9970,2019-01-01 00:15:55.4380,3171,Amsterdam Ave & W 82 St,40.78524672,-73.97667321,3154,E 77 St & 3 Ave,40.77314236,-73.95856158,27451,Subscriber,1987,1
4,2719,2019-01-01 00:07:03.5450,2019-01-01 00:52:22.6500,504,1 Ave & E 16 St,40.73221853,-73.98165557,3709,W 15 St & 6 Ave,40.738046142482766,-73.99642959237099,21579,Subscriber,1990,1


In [278]:
print(df.isna().sum())

tripduration                0
starttime                   0
stoptime                    0
start station id           18
start station name         18
start station latitude      0
start station longitude     0
end station id             18
end station name           18
end station longitude       0
end station latitude        0
bikeid                      0
usertype                    0
birth year                  0
gender                      0
dtype: int64


In [279]:
df = df.dropna()
print(df.shape)

float_list = ['start station latitude','start station longitude','end station latitude','end station longitude']
int_list = ['tripduration', 'start station id', 'end station id','bikeid', 'gender','birth year']

for num_col in (int_list):
    df = df[df[num_col].apply(lambda x: str(x).isnumeric())]
    print(num_col, df.shape)

print(df.shape)

(967270, 15)
tripduration (967269, 15)
start station id (967269, 15)
end station id (967269, 15)
bikeid (967269, 15)
gender (967269, 15)
birth year (967269, 15)
(967269, 15)


In [280]:
df = df.dropna()
a = df.info(memory_usage='deep')


<class 'pandas.core.frame.DataFrame'>
Int64Index: 967269 entries, 1 to 967287
Data columns (total 15 columns):
 #   Column                   Non-Null Count   Dtype 
---  ------                   --------------   ----- 
 0   tripduration             967269 non-null  object
 1   starttime                967269 non-null  object
 2   stoptime                 967269 non-null  object
 3   start station id         967269 non-null  object
 4   start station name       967269 non-null  object
 5   start station latitude   967269 non-null  object
 6   start station longitude  967269 non-null  object
 7   end station id           967269 non-null  object
 8   end station name         967269 non-null  object
 9   end station longitude    967269 non-null  object
 10  end station latitude     967269 non-null  object
 11  bikeid                   967269 non-null  object
 12  usertype                 967269 non-null  object
 13  birth year               967269 non-null  object
 14  gender              

In [281]:
print(df.memory_usage(deep=True).sum()/10**9, "GB")

0.994506052 GB


In [282]:


# for col in int_list:
#     print(f'{col}: min: {df[col].min()}, max: {df[col].max()}')
    
# for col in float_list:
#     print(f'{col}: min: {df[col].min()}, max: {df[col].max()}')

In [283]:
date_cols = ['starttime', 'stoptime']
cat_cols = ['usertype', 'start station name', 'end station name']


def col_type_mod(df_mod):
    for col in int_list:
        df_mod[col] = pd.to_numeric(df_mod[col], downcast='integer')

    for col in float_list:
        df_mod[col] = pd.to_numeric(df_mod[col], downcast='float')
        
    for col in date_cols:
        df_mod[col] = pd.to_datetime(df_mod[col])
        
    for col in cat_cols:
        df_mod[col] = df_mod[col].astype('category')
        
    return df_mod

df = col_type_mod(df)


In [284]:
df.info(memory_usage='deep')

<class 'pandas.core.frame.DataFrame'>
Int64Index: 967269 entries, 1 to 967287
Data columns (total 15 columns):
 #   Column                   Non-Null Count   Dtype         
---  ------                   --------------   -----         
 0   tripduration             967269 non-null  int32         
 1   starttime                967269 non-null  datetime64[ns]
 2   stoptime                 967269 non-null  datetime64[ns]
 3   start station id         967269 non-null  int16         
 4   start station name       967269 non-null  category      
 5   start station latitude   967269 non-null  float32       
 6   start station longitude  967269 non-null  float32       
 7   end station id           967269 non-null  int16         
 8   end station name         967269 non-null  category      
 9   end station longitude    967269 non-null  float32       
 10  end station latitude     967269 non-null  float32       
 11  bikeid                   967269 non-null  int32         
 12  usertype        

In [285]:

for i in range(2,13):
    fname = 'data/bike/csv/2019/2019' + str(i).zfill(2) + '-citibike-tripdata.csv'
    df_temp = pd.read_csv(fname)
    df_org_size = df_temp.memory_usage(index=False, deep=True).sum() / (10**6)

    df_temp = df_temp.dropna()

    for num_col in (int_list):
        df = df[df[num_col].apply(lambda x: str(x).isnumeric())]

    df_mod = col_type_mod(df_temp)    
    df_size = df_mod.memory_usage(index=False, deep=True).sum() / (10**6)
    
    no_rows = df_mod.shape[0]
    obj_col = df_mod.select_dtypes(include=['object']).columns.to_list()
    col_type = df_mod['start station name'].dtype
    print(f'Concatenating month {i}, objects: {col_type}, df rows: {no_rows}, df org size: {round(df_org_size,2)}MB, df shrink size: {round(df_size,2)}MB, row/MB: {round(no_rows/df_size,0)}')
    
    df = pd.concat([df,df_mod], axis=0, ignore_index=True)


Concatenating month 2, objects: category, df rows: 943735, df org size: 435.61MB, df shrink size: 49.23MB, row/MB: 19171.0
Concatenating month 3, objects: category, df rows: 1327950, df org size: 613.27MB, df shrink size: 69.21MB, row/MB: 19188.0
Concatenating month 4, objects: category, df rows: 1766094, df org size: 815.78MB, df shrink size: 92.01MB, row/MB: 19195.0
Concatenating month 5, objects: category, df rows: 1924563, df org size: 889.21MB, df shrink size: 100.27MB, row/MB: 19194.0
Concatenating month 6, objects: category, df rows: 2125370, df org size: 982.61MB, df shrink size: 110.71MB, row/MB: 19198.0
Concatenating month 7, objects: category, df rows: 2181010, df org size: 1007.95MB, df shrink size: 113.59MB, row/MB: 19202.0
Concatenating month 8, objects: category, df rows: 2344135, df org size: 1083.13MB, df shrink size: 122.09MB, row/MB: 19201.0
Concatenating month 9, objects: category, df rows: 2444900, df org size: 1129.3MB, df shrink size: 127.33MB, row/MB: 19202.0
Co

In [286]:
df.info(memory_usage='deep')
df.shape

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 20551517 entries, 0 to 20551516
Data columns (total 15 columns):
 #   Column                   Dtype         
---  ------                   -----         
 0   tripduration             int32         
 1   starttime                datetime64[ns]
 2   stoptime                 datetime64[ns]
 3   start station id         int16         
 4   start station name       object        
 5   start station latitude   float32       
 6   start station longitude  float32       
 7   end station id           int16         
 8   end station name         object        
 9   end station longitude    float32       
 10  end station latitude     float32       
 11  bikeid                   int32         
 12  usertype                 category      
 13  birth year               int16         
 14  gender                   int8          
dtypes: category(1), datetime64[ns](2), float32(4), int16(3), int32(2), int8(1), object(2)
memory usage: 3.9 GB


(20551517, 15)

In [287]:
cat_cols = ['usertype', 'start station name', 'end station name']

def col_type_mod(df_mod):
    for col in cat_cols:
        df_mod[col] = df_mod[col].astype('category')
    
    return df_mod

df = col_type_mod(df)

In [288]:
df.info(memory_usage='deep')


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 20551517 entries, 0 to 20551516
Data columns (total 15 columns):
 #   Column                   Dtype         
---  ------                   -----         
 0   tripduration             int32         
 1   starttime                datetime64[ns]
 2   stoptime                 datetime64[ns]
 3   start station id         int16         
 4   start station name       category      
 5   start station latitude   float32       
 6   start station longitude  float32       
 7   end station id           int16         
 8   end station name         category      
 9   end station longitude    float32       
 10  end station latitude     float32       
 11  bikeid                   int32         
 12  usertype                 category      
 13  birth year               int16         
 14  gender                   int8          
dtypes: category(3), datetime64[ns](2), float32(4), int16(3), int32(2), int8(1)
memory usage: 1019.4 MB


The data is small enoung to process with Pandas. We will explore the data and do some analysis

In [289]:
df.to_parquet(DATA_FOLDER / "pandas_full_2019.parquet")

In [290]:
del df
gc.collect()

74488

In [291]:
df = pd.read_parquet(DATA_FOLDER / "pandas_full_2019.parquet")
print(df.shape)
print(df.memory_usage(deep=True).sum()/10**9, 'GB')
del df
gc.collect()

(20551517, 15)
1.068893769 GB


16